# Обучение модели PERSON (дистилляция) — Colab

Улучшаем распознавание ФИО и убираем ложные срабатывания на топонимах/улицах/орг-названиях. Ученик — лёгкий `rubert-tiny2`.

**Как запускать:** `Runtime → Change runtime type → T4 GPU`, затем `Runtime → Run all` (или ячейки сверху вниз).

Готовую модель в конце скачаешь и пришлёшь на приёмку.

## 1. Получить код и данные
Вариант А — клонировать репозиторий. Если он приватный и клон не проходит — закомментируй Вариант А и раскомментируй Вариант Б (загрузка архива папки `data/training`).

In [ ]:
# --- Вариант А: клонировать репозиторий ---
!git clone https://github.com/EgzodD/ANONIMIZATION_MODULE.git
%cd ANONIMIZATION_MODULE/data/training

# --- Вариант Б: приватный репозиторий -> загрузить training.zip ---
# from google.colab import files
# files.upload()                 # выбери training.zip
# !unzip -q training.zip -d _up
# %cd _up/training               # поправь путь под структуру архива

!ls

## 2. Установить зависимости

In [ ]:
!pip install -q "transformers>=4.40" datasets seqeval spacy
!python -m spacy download ru_core_news_lg

## 3. (по желанию) Перегенерить silver-корпус
`silver/corpus.txt` уже в репозитории. Раскомментируй, если нужно больше данных.

In [ ]:
# !python make_silver_corpus.py 12000
!wc -l silver/corpus.txt

## 4. Разметить silver учителем (spaCy)
spaCy отличает город (LOC) от имени (PER) — это и лечит ложные срабатывания. Проверено: города/улицы/орг → O, имена → PERSON.

In [ ]:
from train_distill import label_silver_with_spacy
label_silver_with_spacy()          # -> silver/corpus.conll
!head -20 silver/corpus.conll

## 5. Обучить ученика `rubert-tiny2`
12 эпох, метрика **F2** (приоритет recall — пропуск ФИО = утечка). Смотри в логах рост `eval_recall`. Результат → `person_ruBERT_distilled/`.

In [ ]:
!python train_distill.py

## 6. Скачать обученную модель
Ячейка сама находит папку модели (train_distill.py сохраняет её по абсолютному пути) — не зависит от текущей директории.

In [ ]:
import glob, subprocess
from google.colab import files
cands = glob.glob('/content/**/person_ruBERT_distilled', recursive=True)
assert cands, 'person_ruBERT_distilled не найдена — обучение не завершилось? Прогони ячейку 5 до конца.'
src = cands[0]; print('нашёл модель:', src)
out = '/content/person_ruBERT_distilled.tar.gz'
subprocess.run(['tar', '-czf', out, '-C', src,
                'config.json', 'model.safetensors',
                'tokenizer.json', 'tokenizer_config.json'], check=True)
print('архив готов:', out)
files.download(out)

## 7. Приёмка
Пришли `person_ruBERT_distilled.tar.gz` — прогоню recall / LeakRate 0% / FP / латентность на held-out `test.jsonl` и, если не хуже старой, промоутну в прод. Веса в git не коммитим.

**Если что-то упало:** OOM → в `train_distill.py` поставь `per_device_train_batch_size=16`; сбросилась среда → повтори шаг 2.